# Autoencoder on Digits Dataset

This notebook implements a 64x16x64 autoencoder to learn a lower-dimensional representation of handwritten digits.

In [8]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sys
sys.path.insert(0, '..')

# Reload module to get latest code
import importlib
if 'nn.nn' in sys.modules:
    importlib.reload(sys.modules['nn.nn'])

from nn.nn import NeuralNetwork

## Step 1: Load and prepare data

In [3]:
# Load digits dataset
digits = load_digits()
X = digits.data
print(f"Dataset shape: {X.shape}")
print(f"Feature range: [{X.min()}, {X.max()}]")

Dataset shape: (1797, 64)
Feature range: [0.0, 16.0]


In [4]:
# Normalize the data to [0, 1]
X = X / X.max()

# Split into training and validation sets
X_train, X_val = train_test_split(X, test_size=0.2, random_state=42)
print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")

Training set shape: (1437, 64)
Validation set shape: (360, 64)


## Step 2: Create and configure autoencoder

In [5]:
# Define autoencoder architecture: 64 -> 16 -> 64
# For autoencoder, the target is to reconstruct the input, so y = X
nn_arch = [
    {'input_dim': 64, 'output_dim': 16, 'activation': 'relu'},
    {'input_dim': 16, 'output_dim': 64, 'activation': 'sigmoid'}
]

# Create the autoencoder
autoencoder = NeuralNetwork(
    nn_arch=nn_arch,
    lr=0.01,
    seed=42,
    batch_size=32,
    epochs=100,
    loss_function='mean_squared_error'
)

print("Autoencoder created successfully")

Autoencoder created successfully


## Step 3: Train the autoencoder

In [9]:
# For autoencoder, the target is to reconstruct the input
y_train = X_train
y_val = X_val

# Train the autoencoder
train_losses, val_losses = autoencoder.fit(X_train, y_train, X_val, y_val)

print(f"Training completed")
print(f"Final training loss: {train_losses[-1]:.6f}")
print(f"Final validation loss: {val_losses[-1]:.6f}")

ValueError: operands could not be broadcast together with shapes (64,32) (1437,64) 

## Step 4: Plot training and validation loss

In [ ]:
# Plot training and validation loss
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss', linewidth=2)
plt.plot(val_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss (MSE)', fontsize=12)
plt.title('Autoencoder Training and Validation Loss', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 5: Quantify reconstruction error on validation set

In [ ]:
# Get reconstructions on validation set
y_hat_val = autoencoder.predict(X_val)

# Calculate reconstruction error
reconstruction_error = np.mean((X_val - y_hat_val) ** 2)
print(f"Average Reconstruction Error (MSE) on Validation Set: {reconstruction_error:.6f}")

# Per-sample reconstruction error
per_sample_error = np.mean((X_val - y_hat_val) ** 2, axis=1)
print(f"Mean per-sample error: {np.mean(per_sample_error):.6f}")
print(f"Std per-sample error: {np.std(per_sample_error):.6f}")
print(f"Min per-sample error: {np.min(per_sample_error):.6f}")
print(f"Max per-sample error: {np.max(per_sample_error):.6f}")

## Step 6: Visualize reconstructions

In [ ]:
# Visualize some original vs reconstructed digits
fig, axes = plt.subplots(4, 10, figsize=(15, 6))

for i in range(20):
    # Original
    axes[2*i//10, (2*i)%10].imshow(X_val[i].reshape(8, 8), cmap='gray')
    axes[2*i//10, (2*i)%10].set_title('Original', fontsize=8)
    axes[2*i//10, (2*i)%10].axis('off')
    
    # Reconstructed
    axes[(2*i+1)//10, (2*i+1)%10].imshow(y_hat_val[i].reshape(8, 8), cmap='gray')
    axes[(2*i+1)//10, (2*i+1)%10].set_title('Reconstructed', fontsize=8)
    axes[(2*i+1)//10, (2*i+1)%10].axis('off')

plt.tight_layout()
plt.show()

## Hyperparameter Explanation

### Chosen Hyperparameters:
- **Learning Rate (lr=0.01)**: A moderate learning rate that balances convergence speed and stability
- **Batch Size (32)**: A standard batch size that works well for the digits dataset (1797 samples)
- **Epochs (100)**: Enough iterations to allow the network to learn without overfitting
- **Loss Function (MSE)**: Mean Squared Error is appropriate for reconstruction tasks
- **Activation Functions**:
  - Hidden layer: ReLU provides non-linearity and helps with deep learning
  - Output layer: Sigmoid constrains output to [0,1] range, matching normalized input

### Architecture Rationale:
- **64 → 16**: Compresses the 64 input features into a 16-dimensional latent space (75% compression)
- **16 → 64**: Reconstructs the original 64-dimensional space from the latent representation
- This bottleneck forces the network to learn compact representations of the digits

### Why These Choices Work:
1. The relatively small dataset benefits from moderate learning rate and batch size
2. MSE loss is standard for reconstruction and pixel-level tasks
3. The 16-dimensional latent space provides a good balance between compression and reconstruction quality
4. 100 epochs is sufficient for convergence on this relatively simple dataset